## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [1]:
import tensorflow as tf

In [71]:
tf.__version__

'2.0.0-beta1'

In [0]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default


In [72]:
tf.executing_eagerly()

True

### Collect Data

In [0]:
from google.colab import drive
drive.mount('/gdrive')

In [2]:
import numpy as np
import pandas as pd

In [246]:
price_data = pd.read_csv('prices.csv')


In [247]:
price_data.head()

,date,symbol,open,close,low,high,volume
0,2016-01-05 00:00:00,WLTW,123.430000,125.839996,122.309998,126.250000,2163600.0
1,2016-01-06 00:00:00,WLTW,125.239998,119.980003,119.940002,125.540001,2386400.0
2,2016-01-07 00:00:00,WLTW,116.379997,114.949997,114.930000,119.739998,2489500.0
3,2016-01-08 00:00:00,WLTW,115.480003,116.620003,113.500000,117.440002,2006300.0
4,2016-01-11 00:00:00,WLTW,117.010002,114.970001,114.089996,117.330002,1408600.0


### Check all columns in the dataset

In [248]:
price_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 851264 entries, 0 to 851263
Data columns (total 7 columns):
date      851264 non-null object
symbol    851264 non-null object
open      851264 non-null float64
close     851264 non-null float64
low       851264 non-null float64
high      851264 non-null float64
volume    851264 non-null float64
dtypes: float64(5), object(2)
memory usage: 45.5+ MB


### Drop columns `date` and  `symbol`

In [249]:
price_data = price_data.drop(["date","symbol"], axis = 1)

In [250]:
price_data.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [251]:
price_df=price_data.head(1000)

In [252]:
print("Shape of the updated dataframe : {0}".format(price_df.shape))

Shape of the updated dataframe : (1000, 5)


In [253]:
price_df["volume"]=price_df["volume"]/1000000

C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """Entry point for launching an IPython kernel.


### Divide the data into train and test sets

In [254]:
from sklearn.model_selection import train_test_split

In [255]:
#Split dataset into Train (80%) and Test (20%)
X =  price_df.drop("volume", axis=1)
Y =  price_df.pop("volume")
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=7)

In [256]:
X_train.shape

(800, 4)

In [257]:
X_test.shape

(200, 4)

#### Convert Training and Test Data to numpy float32 arrays


In [258]:
X_train = np.asarray(X_train, dtype='float32')
X_test = np.asarray(X_test, dtype='float32')
y_train = np.asarray(y_train, dtype='float32')
y_test = np.asarray(y_test, dtype='float32')

In [259]:
type(y_train)

numpy.ndarray

In [260]:
type(X_train)

numpy.ndarray

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [261]:
from sklearn.preprocessing import Normalizer

In [262]:
X_train=Normalizer().fit_transform(X_train)

In [263]:
X_test=Normalizer().fit_transform(X_test)

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [264]:
W = tf.Variable(tf.random.normal(shape=[4,1]), name="Weights")
b = tf.Variable(tf.random.normal(shape=[1]),name="Bias")

2.Define a function to calculate prediction

In [265]:
def fn_prediction(x,W,b):
    return tf.add(tf.matmul(x, W),b,name='output')
    

3.Loss (Cost) Function [Mean square error]

In [266]:
y_pred=fn_prediction(X_train,W,b)

In [267]:
def fn_loss(y,y_):
    return tf.reduce_mean(tf.square(y-y_),name='Loss')

In [268]:
loss = fn_loss(y_train,y_pred)

In [269]:
print(loss)

tf.Tensor(275.69702, shape=(), dtype=float32)


4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [270]:
def fn_train(x,y,W,b,learn_rate):
  with tf.GradientTape() as t :
     t.watch([W,b])
     y_pred = fn_prediction(x,W,b)
     loss = fn_loss(y,y_pred)
  
  dc_dw, dc_db = t.gradient(loss, [W, b])
  W = W - learn_rate * dc_dw
  b = b - learn_rate * dc_db
  return W,b

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [271]:
for i in range (100):
  X_train=np.asarray(X_train)
  W,b = fn_train(X_train,y_train,W,b,0.01)
  y_pred = fn_prediction(X_train,W,b)
  train_loss = fn_loss(y_train,y_pred)
  if i % 5 == 0:
    print('Training loss at step: ', i, ' is ', train_loss)

Training loss at step:  0  is  tf.Tensor(273.27106, shape=(), dtype=float32)
Training loss at step:  5  is  tf.Tensor(263.7128, shape=(), dtype=float32)
Training loss at step:  10  is  tf.Tensor(257.3581, shape=(), dtype=float32)
Training loss at step:  15  is  tf.Tensor(253.13322, shape=(), dtype=float32)
Training loss at step:  20  is  tf.Tensor(250.32437, shape=(), dtype=float32)
Training loss at step:  25  is  tf.Tensor(248.45697, shape=(), dtype=float32)
Training loss at step:  30  is  tf.Tensor(247.2154, shape=(), dtype=float32)
Training loss at step:  35  is  tf.Tensor(246.39, shape=(), dtype=float32)
Training loss at step:  40  is  tf.Tensor(245.84125, shape=(), dtype=float32)
Training loss at step:  45  is  tf.Tensor(245.47638, shape=(), dtype=float32)
Training loss at step:  50  is  tf.Tensor(245.2338, shape=(), dtype=float32)
Training loss at step:  55  is  tf.Tensor(245.07256, shape=(), dtype=float32)
Training loss at step:  60  is  tf.Tensor(244.96533, shape=(), dtype=floa

### Get the shapes and values of W and b

In [272]:
W

<tf.Tensor: id=5146971, shape=(4, 1), dtype=float32, numpy=
array([[0.9323867 ],
       [0.16453052],
       [2.15604   ],
       [1.0950297 ]], dtype=float32)>

In [273]:
W.shape

TensorShape([4, 1])

In [274]:
b

<tf.Tensor: id=5146974, shape=(1,), dtype=float32, numpy=array([3.1330466], dtype=float32)>

In [275]:
b.shape

TensorShape([1])

### Model Prediction on 1st Examples in Test Dataset

In [277]:
y_test_pred = fn_prediction(X_test, W, b)
test_loss = fn_loss(y_test, y_test_pred)

In [278]:
test_loss

<tf.Tensor: id=5146995, shape=(), dtype=float32, numpy=66.18423>

## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

In [297]:
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam

Using TensorFlow backend.


### Load the given Iris data using pandas (Iris.csv)

In [286]:
iris_data = pd.read_csv('Iris.csv')

In [287]:
iris_data.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

In [288]:
iris_df = pd.get_dummies(iris_data,columns=["Species"], drop_first=False)

In [289]:
iris_df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species_Iris-setosa,Species_Iris-versicolor,Species_Iris-virginica
0,1,5.1,3.5,1.4,0.2,1,0,0
1,2,4.9,3.0,1.4,0.2,1,0,0
2,3,4.7,3.2,1.3,0.2,1,0,0
3,4,4.6,3.1,1.5,0.2,1,0,0
4,5,5.0,3.6,1.4,0.2,1,0,0


### Splitting the data into feature set and target set

In [290]:
#Split dataset into Train (80%) and Test (20%)
feature =  iris_df[["SepalLengthCm","SepalWidthCm","PetalLengthCm","PetalWidthCm"]]
target =  iris_df[["Species_Iris-setosa","Species_Iris-versicolor","Species_Iris-virginica"]]


In [292]:
feature.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [293]:
target.head()

,Species_Iris-setosa,Species_Iris-versicolor,Species_Iris-virginica
0,1,0,0
1,1,0,0
2,1,0,0
3,1,0,0
4,1,0,0


In [304]:
train_X, test_X, train_y, test_y = train_test_split(feature, target, test_size=0.20)

In [305]:
train_X.shape

(120, 4)

In [306]:
test_X.shape

(30, 4)

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [315]:
model = tf.keras.Sequential([
  tf.keras.layers.Dense(10, activation=tf.nn.relu, input_shape=(4,)),  # input shape required
  tf.keras.layers.Dense(3,activation=tf.nn.softmax, name='output')
])

In [316]:
model.compile(optimizer=tf.keras.optimizers.SGD(), loss='categorical_crossentropy', metrics=['accuracy'])

In [317]:
print('Neural Network Model Summary: ')
print(model.summary())

Neural Network Model Summary: 
Model: "sequential_2"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_4 (Dense)              (None, 10)                50        
_________________________________________________________________
output (Dense)               (None, 3)                 33        
Total params: 83
Trainable params: 83
Non-trainable params: 0
_________________________________________________________________
None


### Model Training 

In [318]:
model.fit(train_X, train_y,epochs=200)

Train on 120 samples
Epoch 1/200
120/120 [==============================] - 1s 4ms/sample - loss: 2.2772 - accuracy: 0.3500
Epoch 2/200
120/120 [==============================] - 0s 217us/sample - loss: 1.1837 - accuracy: 0.2500
Epoch 3/200
120/120 [==============================] - 0s 250us/sample - loss: 1.0633 - accuracy: 0.3583
Epoch 4/200
120/120 [==============================] - 0s 408us/sample - loss: 1.0041 - accuracy: 0.3750
Epoch 5/200
120/120 [==============================] - 0s 317us/sample - loss: 0.9569 - accuracy: 0.5750
Epoch 6/200
120/120 [==============================] - 0s 217us/sample - loss: 0.9132 - accuracy: 0.6917
Epoch 7/200
120/120 [==============================] - 0s 267us/sample - loss: 0.8735 - accuracy: 0.6917
Epoch 8/200
120/120 [==============================] - 0s 250us/sample - loss: 0.8432 - accuracy: 0.6917
Epoch 9/200
120/120 [==============================] - 0s 317us/sample - loss: 0.8092 - accuracy: 0.6917
Epoch 10/200
120/120 [==============

120/120 [==============================] - 0s 142us/sample - loss: 0.3882 - accuracy: 0.8917
Epoch 79/200
120/120 [==============================] - 0s 167us/sample - loss: 0.3845 - accuracy: 0.8833
Epoch 80/200
120/120 [==============================] - 0s 133us/sample - loss: 0.3824 - accuracy: 0.9417
Epoch 81/200
120/120 [==============================] - 0s 167us/sample - loss: 0.3809 - accuracy: 0.9167
Epoch 82/200
120/120 [==============================] - 0s 133us/sample - loss: 0.3794 - accuracy: 0.9417
Epoch 83/200
120/120 [==============================] - 0s 150us/sample - loss: 0.3778 - accuracy: 0.9417
Epoch 84/200
120/120 [==============================] - 0s 150us/sample - loss: 0.3784 - accuracy: 0.8833
Epoch 85/200
120/120 [==============================] - 0s 133us/sample - loss: 0.3746 - accuracy: 0.8583
Epoch 86/200
120/120 [==============================] - 0s 175us/sample - loss: 0.3718 - accuracy: 0.9667
Epoch 87/200
120/120 [==============================] - 0s 

120/120 [==============================] - 0s 100us/sample - loss: 0.2666 - accuracy: 0.9667
Epoch 155/200
120/120 [==============================] - 0s 108us/sample - loss: 0.2659 - accuracy: 0.9667
Epoch 156/200
120/120 [==============================] - 0s 100us/sample - loss: 0.2656 - accuracy: 0.9667
Epoch 157/200
120/120 [==============================] - 0s 83us/sample - loss: 0.2621 - accuracy: 0.9667
Epoch 158/200
120/120 [==============================] - 0s 100us/sample - loss: 0.2610 - accuracy: 0.9667
Epoch 159/200
120/120 [==============================] - 0s 100us/sample - loss: 0.2649 - accuracy: 0.9750
Epoch 160/200
120/120 [==============================] - 0s 92us/sample - loss: 0.2596 - accuracy: 0.9667
Epoch 161/200
120/120 [==============================] - 0s 133us/sample - loss: 0.2622 - accuracy: 0.9667
Epoch 162/200
120/120 [==============================] - 0s 142us/sample - loss: 0.2566 - accuracy: 0.9667
Epoch 163/200
120/120 [==============================

### Model Prediction

In [319]:
results = model.evaluate(test_X, test_y)

30/30 [==============================] - 0s 2ms/sample - loss: 0.2668 - accuracy: 0.9667


In [320]:
print('Final test set loss: {:4f}'.format(results[0]))
print('Final test set accuracy: {:4f}'.format(results[1]))

Final test set loss: 0.266777
Final test set accuracy: 0.966667


### Save the Model

In [321]:
model.save('iris_model.hs')

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?

In [328]:
model_2layer = tf.keras.Sequential([
  tf.keras.layers.Dense(20, activation=tf.nn.relu, input_shape=(4,)),  # input shape required
  tf.keras.layers.Dense(20, activation=tf.nn.relu),
  tf.keras.layers.Dense(3,activation=tf.nn.softmax, name='output')
])

In [329]:
model_2layer.compile(optimizer=tf.keras.optimizers.SGD(), loss='categorical_crossentropy', metrics=['accuracy'])

In [330]:
print('Neural Network Model Summary: ')
print(model_2layer.summary())

Neural Network Model Summary: 
Model: "sequential_4"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_7 (Dense)              (None, 20)                100       
_________________________________________________________________
dense_8 (Dense)              (None, 20)                420       
_________________________________________________________________
output (Dense)               (None, 3)                 63        
Total params: 583
Trainable params: 583
Non-trainable params: 0
_________________________________________________________________
None


In [331]:
model_2layer.fit(train_X, train_y,epochs=200)

Train on 120 samples
Epoch 1/200
120/120 [==============================] - 0s 3ms/sample - loss: 1.7897 - accuracy: 0.3083
Epoch 2/200
120/120 [==============================] - 0s 133us/sample - loss: 1.2260 - accuracy: 0.3083
Epoch 3/200
120/120 [==============================] - 0s 258us/sample - loss: 1.0095 - accuracy: 0.4417
Epoch 4/200
120/120 [==============================] - 0s 158us/sample - loss: 0.9238 - accuracy: 0.7083
Epoch 5/200
120/120 [==============================] - 0s 225us/sample - loss: 0.8787 - accuracy: 0.9083
Epoch 6/200
120/120 [==============================] - 0s 200us/sample - loss: 0.8538 - accuracy: 0.7417
Epoch 7/200
120/120 [==============================] - 0s 175us/sample - loss: 0.8312 - accuracy: 0.8083
Epoch 8/200
120/120 [==============================] - 0s 150us/sample - loss: 0.8120 - accuracy: 0.7083
Epoch 9/200
120/120 [==============================] - 0s 183us/sample - loss: 0.7990 - accuracy: 0.7333
Epoch 10/200
120/120 [==============

120/120 [==============================] - 0s 117us/sample - loss: 0.2888 - accuracy: 0.9833
Epoch 77/200
120/120 [==============================] - 0s 133us/sample - loss: 0.2853 - accuracy: 0.9667
Epoch 78/200
120/120 [==============================] - 0s 133us/sample - loss: 0.2817 - accuracy: 0.9667
Epoch 79/200
120/120 [==============================] - 0s 158us/sample - loss: 0.2833 - accuracy: 0.9667
Epoch 80/200
120/120 [==============================] - 0s 150us/sample - loss: 0.2747 - accuracy: 0.9667
Epoch 81/200
120/120 [==============================] - 0s 100us/sample - loss: 0.2763 - accuracy: 0.9667
Epoch 82/200
120/120 [==============================] - 0s 92us/sample - loss: 0.2703 - accuracy: 0.9667
Epoch 83/200
120/120 [==============================] - 0s 100us/sample - loss: 0.2666 - accuracy: 0.9750
Epoch 84/200
120/120 [==============================] - 0s 133us/sample - loss: 0.2671 - accuracy: 0.9667
Epoch 85/200
120/120 [==============================] - 0s 1

120/120 [==============================] - 0s 100us/sample - loss: 0.1526 - accuracy: 0.9667
Epoch 154/200
120/120 [==============================] - 0s 67us/sample - loss: 0.1529 - accuracy: 0.9667
Epoch 155/200
120/120 [==============================] - 0s 75us/sample - loss: 0.1476 - accuracy: 0.9667
Epoch 156/200
120/120 [==============================] - 0s 100us/sample - loss: 0.1470 - accuracy: 0.9750
Epoch 157/200
120/120 [==============================] - 0s 100us/sample - loss: 0.1476 - accuracy: 0.9833
Epoch 158/200
120/120 [==============================] - 0s 67us/sample - loss: 0.1509 - accuracy: 0.9750
Epoch 159/200
120/120 [==============================] - 0s 75us/sample - loss: 0.1420 - accuracy: 0.9833
Epoch 160/200
120/120 [==============================] - 0s 100us/sample - loss: 0.1410 - accuracy: 0.9833
Epoch 161/200
120/120 [==============================] - 0s 108us/sample - loss: 0.1422 - accuracy: 0.9750
Epoch 162/200
120/120 [==============================] 

In [332]:
results = model_2layer.evaluate(test_X, test_y)

30/30 [==============================] - 0s 5ms/sample - loss: 0.1601 - accuracy: 0.9667


In [333]:
print('Final test set loss: {:4f}'.format(results[0]))
print('Final test set accuracy: {:4f}'.format(results[1]))

Final test set loss: 0.160073
Final test set accuracy: 0.966667
